In [14]:
from pathlib import Path
import os

# ── Paths ─────────────────────────────────────────────────────────────────────
BASE_PATH      = r"/nexus/posix0/MBR-neuralsystems/ephys/GLC7248/GLC7248_a_g1/preprocessed"
PEAKS_DIR      = os.path.join(BASE_PATH, "thresholding_peaks_3.5")
PEAK_LOCS_PATH = os.path.join(PEAKS_DIR, "localized_peaks_mt_raw.npy")
MOTION_DIR     = os.path.join(PEAKS_DIR, "motion_estimates", "motion_dredge")
PLOTS_DIR      = os.path.join(PEAKS_DIR, "spike_density_plots_3.5motion")   # if None → PEAKS_DIR / "spike_density_plots"

# CLUSTERS_PATH = None
CLUSTERS_PATH  = r"/nexus/posix0/MBR-neuralsystems/ephys/GLC7248/GLC7248_a_g1/GLC7248_a_g1_imec0/clusters.npz"

# ── Section 1: Waveform snippet windows ──────────────────────────────────────
WAVEFORM_WINDOWS = [
    {"plot_start_s": 100, "channel_start": 0, "plot_duration": 0.05, "channel_end": 384},
    {"plot_start_s": 600,  "channel_start": 0, "plot_duration": 0.05, "channel_end": 384},
    {"plot_start_s": 1100,  "channel_start": 0, "plot_duration": 0.05, "channel_end": 384},
]

# ── Section 2 & 3: Scatter / motion-overlay windows ──────────────────────────
SCATTER_WINDOWS    = None
SCATTER_DURATION_S = 720.0

# ── Analysis time range ───────────────────────────────────────────────────────
ANALYSIS_START_S = None
ANALYSIS_END_S   = None

# ── Normalisation ─────────────────────────────────────────────────────────────
NORMALISE_BY_DURATION = True   # True → spikes / s;  False → raw counts

# ── Misc ──────────────────────────────────────────────────────────────────────
PLOT_DPI          = 150
RNG_SEED          = 42
CORRECTION_METHOD = "dredge"
RMS_DURATION_S    = 30 * 60
RMS_START_S       = 0.0

In [15]:
import json
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as mticker
from matplotlib.patches import Rectangle
import spikeinterface as si


In [3]:
def _time_mask(arr_times, t0, t1):
    return (arr_times >= t0) & (arr_times < t1)

# ── Resolve paths ─────────────────────────────────────────────────────────────
BASE_PATH  = Path(BASE_PATH)
PEAKS_DIR  = Path(PEAKS_DIR)

PLOTS_DIR  = Path(PLOTS_DIR)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

_locs_path = Path(PEAK_LOCS_PATH)
HAS_LOCS   = _locs_path.exists()

MOTION_DIR = Path(MOTION_DIR)
HAS_MOTION = MOTION_DIR is not None

if CLUSTERS_PATH is not None:
    _clusters_path = Path(CLUSTERS_PATH)
    HAS_CLUSTERS = _clusters_path.exists()

print("="*70)
print("SPIKE DENSITY PIPELINE")
print("="*70)
print(f"  Base path  : {BASE_PATH}")
print(f"  Peaks dir  : {PEAKS_DIR}")
print(f"  Locs file  : {_locs_path}  {'✓' if HAS_LOCS else '✗ NOT FOUND — falling back to channel-map positions'}")
print(f"  Motion dir : {MOTION_DIR if HAS_MOTION else '(none — displacement = 0)'}")
print(f"  Clusters   : {_clusters_path if HAS_CLUSTERS else '✗ NOT FOUND — sorted-units panel will be left blank'}")
print(f"  Plots dir  : {PLOTS_DIR}")

SPIKE DENSITY PIPELINE
  Base path  : /nexus/posix0/MBR-neuralsystems/ephys/GLC6962/GLC6962_q_g0/preprocessed
  Peaks dir  : /nexus/posix0/MBR-neuralsystems/ephys/GLC6962/GLC6962_q_g0/preprocessed/thresholding_peaks_3.5
  Locs file  : /nexus/posix0/MBR-neuralsystems/ephys/GLC6962/GLC6962_q_g0/preprocessed/thresholding_peaks_3.5/localized_peaks_mt_raw.npy  ✓
  Motion dir : /nexus/posix0/MBR-neuralsystems/ephys/GLC6962/GLC6962_q_g0/preprocessed/thresholding_peaks_3.5/motion_estimates/motion_dredge
  Clusters   : ✗ NOT FOUND — sorted-units panel will be left blank
  Plots dir  : /nexus/posix0/MBR-neuralsystems/ephys/GLC6962/GLC6962_q_g0/preprocessed/thresholding_peaks_3.5/spike_density_plots_3.5motion


In [11]:
# ── Load recording ────────────────────────────────────────────────────────────
print("\n[1/6] Loading recording …")
rec = si.load(str(BASE_PATH))
fs          = rec.sampling_frequency
t_total     = rec.get_total_duration()
time_offset = float(rec.get_times()[0])
n_channels  = rec.get_num_channels()
print(n_channels)
channel_positions = rec.get_channel_locations()
print(channel_positions)
channel_depths    = channel_positions[:, 1]
print(channel_depths)
sort_idx_ch       = np.argsort(channel_depths)
print(sort_idx_ch)
sorted_depths_ch  = channel_depths[sort_idx_ch]
print(sorted_depths_ch)
print(f"  Duration : {t_total:.1f} s   |  fs = {fs:.0f} Hz  |  channels = {n_channels}")


[1/6] Loading recording …
357
[[  16.    0.]
 [  48.    0.]
 [   0.   20.]
 [  32.   20.]
 [  16.   40.]
 [  48.   40.]
 [   0.   60.]
 [  32.   60.]
 [  16.   80.]
 [  48.   80.]
 [  32.  100.]
 [  16.  120.]
 [  48.  120.]
 [   0.  140.]
 [  32.  140.]
 [  16.  160.]
 [  48.  160.]
 [   0.  180.]
 [  32.  180.]
 [  16.  200.]
 [  48.  200.]
 [   0.  220.]
 [  32.  220.]
 [  16.  240.]
 [  48.  240.]
 [   0.  260.]
 [  32.  260.]
 [  16.  280.]
 [  48.  280.]
 [   0.  300.]
 [  32.  300.]
 [  16.  320.]
 [  48.  320.]
 [  32.  340.]
 [  16.  360.]
 [  48.  360.]
 [   0.  380.]
 [  32.  380.]
 [  16.  400.]
 [  48.  400.]
 [   0.  420.]
 [  32.  420.]
 [  16.  440.]
 [  48.  440.]
 [   0.  460.]
 [  32.  460.]
 [  16.  480.]
 [  48.  480.]
 [   0.  500.]
 [  32.  500.]
 [  16.  520.]
 [  48.  520.]
 [   0.  540.]
 [  16.  560.]
 [   0.  580.]
 [  16.  600.]
 [  48.  600.]
 [   0.  620.]
 [  32.  620.]
 [  16.  640.]
 [  48.  640.]
 [   0.  660.]
 [  32.  660.]
 [  16.  680.]
 [  48.  

In [12]:
print(f"  Channel IDs : {rec.get_channel_ids()}")

  Channel IDs : ['imec0.ap#AP0' 'imec0.ap#AP1' 'imec0.ap#AP2' 'imec0.ap#AP3'
 'imec0.ap#AP4' 'imec0.ap#AP5' 'imec0.ap#AP6' 'imec0.ap#AP7'
 'imec0.ap#AP8' 'imec0.ap#AP9' 'imec0.ap#AP11' 'imec0.ap#AP12'
 'imec0.ap#AP13' 'imec0.ap#AP14' 'imec0.ap#AP15' 'imec0.ap#AP16'
 'imec0.ap#AP17' 'imec0.ap#AP18' 'imec0.ap#AP19' 'imec0.ap#AP20'
 'imec0.ap#AP21' 'imec0.ap#AP22' 'imec0.ap#AP23' 'imec0.ap#AP24'
 'imec0.ap#AP25' 'imec0.ap#AP26' 'imec0.ap#AP27' 'imec0.ap#AP28'
 'imec0.ap#AP29' 'imec0.ap#AP30' 'imec0.ap#AP31' 'imec0.ap#AP32'
 'imec0.ap#AP33' 'imec0.ap#AP35' 'imec0.ap#AP36' 'imec0.ap#AP37'
 'imec0.ap#AP38' 'imec0.ap#AP39' 'imec0.ap#AP40' 'imec0.ap#AP41'
 'imec0.ap#AP42' 'imec0.ap#AP43' 'imec0.ap#AP44' 'imec0.ap#AP45'
 'imec0.ap#AP46' 'imec0.ap#AP47' 'imec0.ap#AP48' 'imec0.ap#AP49'
 'imec0.ap#AP50' 'imec0.ap#AP51' 'imec0.ap#AP52' 'imec0.ap#AP53'
 'imec0.ap#AP54' 'imec0.ap#AP56' 'imec0.ap#AP58' 'imec0.ap#AP60'
 'imec0.ap#AP61' 'imec0.ap#AP62' 'imec0.ap#AP63' 'imec0.ap#AP64'
 'imec0.ap#AP65' 'i

In [8]:
print("\n[1b/6] Loading threshold params and noise levels …")
_params_path = PEAKS_DIR / "params.json"
if _params_path.exists():
    with open(_params_path) as _f:
        _params = json.load(_f)
    detect_threshold = float(_params.get("detect_threshold"))
print(f"  detect_threshold = {detect_threshold}")


[1b/6] Loading threshold params and noise levels …
  detect_threshold = 3.5


In [9]:
noise_levels = si.get_noise_levels(rec, return_scaled=False)
abs_thresholds = noise_levels * detect_threshold   # (n_channels,)
print(f"  Noise levels range: {noise_levels.min():.2f} – {noise_levels.max():.2f}")
print(f"  Threshold range  : {abs_thresholds.min():.2f} – {abs_thresholds.max():.2f}")

noise_level (no parallelization): 100%|██████████| 20/20 [00:08<00:00,  2.37it/s]

  Noise levels range: 0.64 – 1.68
  Threshold range  : 2.25 – 5.87


In [10]:
print("\n[1c/6] Parsing channel numbers from channel_ids for grid placement …")
import re as _re
_chan_id_strs = [str(c) for c in rec.get_channel_ids()]

def _parse_channel_number(s):
    m = _re.search(r"(\d+)$", s)
    if m is None:
        raise ValueError(f"Could not parse a trailing channel number from channel_id '{s}'")
    return int(m.group(1))

chan_num_local = np.array([_parse_channel_number(s) for s in _chan_id_strs], dtype=int)
n_total_channels = int(chan_num_local.max()) + 1   # assumes numbering starts at 0
present_set = set(chan_num_local.tolist())
deleted_channel_numbers = sorted(set(range(n_total_channels)) - present_set)
print(f"  Parsed {n_channels} channel numbers, range 0–{n_total_channels - 1} expected.")
print(f"  Deleted/missing channel numbers ({len(deleted_channel_numbers)}): {deleted_channel_numbers}")



[1c/6] Parsing channel numbers from channel_ids for grid placement …
  Parsed 357 channel numbers, range 0–383 expected.
  Deleted/missing channel numbers (27): [10, 34, 55, 57, 59, 81, 95, 97, 148, 155, 172, 174, 191, 200, 213, 248, 283, 311, 313, 326, 331, 345, 354, 358, 360, 362, 378]


In [13]:
CLUSTERS_PATH  = r"GLC6811/GLC6811__g2/GLC6811__g2_imec0/clusters.npz"
if CLUSTERS_PATH is not None:
    _clusters_path = Path(CLUSTERS_PATH)
    HAS_CLUSTERS = _clusters_path.exists()

print("="*70)
print("SPIKE DENSITY PIPELINE")
print("="*70)
print(f"  Base path  : {BASE_PATH}")
print(f"  Peaks dir  : {PEAKS_DIR}")
print(f"  Locs file  : {_locs_path}  {'✓' if HAS_LOCS else '✗ NOT FOUND — falling back to channel-map positions'}")
print(f"  Motion dir : {MOTION_DIR if HAS_MOTION else '(none — displacement = 0)'}")
print(f"  Clusters   : {_clusters_path if HAS_CLUSTERS else '✗ NOT FOUND — sorted-units panel will be left blank'}")
print(f"  Plots dir  : {PLOTS_DIR}")

SPIKE DENSITY PIPELINE
  Base path  : /nexus/posix0/MBR-neuralsystems/ephys/GLC6962/GLC6962_q_g0/preprocessed
  Peaks dir  : /nexus/posix0/MBR-neuralsystems/ephys/GLC6962/GLC6962_q_g0/preprocessed/thresholding_peaks_3.5
  Locs file  : /nexus/posix0/MBR-neuralsystems/ephys/GLC6962/GLC6962_q_g0/preprocessed/thresholding_peaks_3.5/localized_peaks_mt_raw.npy  ✓
  Motion dir : /nexus/posix0/MBR-neuralsystems/ephys/GLC6962/GLC6962_q_g0/preprocessed/thresholding_peaks_3.5/motion_estimates/motion_dredge
  Clusters   : ✗ NOT FOUND — sorted-units panel will be left blank
  Plots dir  : /nexus/posix0/MBR-neuralsystems/ephys/GLC6962/GLC6962_q_g0/preprocessed/thresholding_peaks_3.5/spike_density_plots_3.5motion


In [23]:
def plot_spikes_scatter(peaks, peak_locations, sampling_frequency, ax=None, 
                       t_start=None, t_end=None, title="Peak Depths Over Time", **scatter_kws):
    """Common scatter plot for peaks over time (used by multiple plot functions).

    NOTE: peaks['sample_index'] is expected to already be relative to the
    ORIGINAL (unsliced) recording, so t_start/t_end here are in original-
    recording time, matching SCATTER_WINDOW_S windows etc.
    """
    if ax is None:
        ax = plt.gca()
    
    # Apply time mask if specified
    if t_start is not None or t_end is not None:
        times = peaks['sample_index'] / sampling_frequency
        mask = np.ones(len(peaks), dtype=bool)
        if t_start is not None:
            mask &= (times >= t_start)
        if t_end is not None:
            mask &= (times <= t_end)
        peaks_plot = peaks[mask]
        locations_plot = peak_locations[mask]
    else:
        peaks_plot = peaks
        locations_plot = peak_locations
    
    scatter_defaults = {'s': 1, 'color': 'k', 'alpha': 0.01}
    scatter_defaults.update(scatter_kws)
    
    ax.scatter(peaks_plot['sample_index'] / sampling_frequency, 
               locations_plot['y'], 
               **scatter_defaults)
    ax.set_xlabel('Time (s)', fontsize=12)
    ax.set_ylabel('Depth (µm)', fontsize=12)
    ax.set_title(title, fontsize=14)
    ax.grid(True, alpha=0.2)
    return ax

In [25]:
peaks = np.load(r"/nexus/posix0/MBR-neuralsystems/ephys/GLC7248/GLC7248_a_g1/preprocessed/thresholding_peaks_3.5_ts5600_teend/_sliced_recording/thresholding_peaks_3.5_ts5600_teend/peaks_combined2_sorted.npy")
print(peaks)


[(168000064,  37, -6.0039062, 0, 0, nan)
 (168000074,  70, -3.8320312, 0, 0, nan)
 (168000141, 163, -3.1738281, 0, 0, nan) ...
 (292621911,  37, -5.9140625, 0, 0, nan)
 (292622076,  98, -7.2617188, 0, 0, nan)
 (292622348,  75, -4.34375  , 0, 0, nan)]
